# Phân loại độ chín quả chuối trên Kaggle bằng 5 mô hình

Notebook này dùng cho bài toán ba lớp:

- `under ripe / unripe` → **Xanh**
- `ripe` → **Chín**
- `past ripe / overripe` → **Quá chín**

Năm mô hình được so sánh trên **cùng một cách chia dữ liệu**:

1. MobileNetV3Small
2. EfficientNetB0
3. ResNet50
4. DenseNet121
5. InceptionV3

## Trước khi chạy

1. Mở Kaggle Notebook và chọn **Add Input** để thêm dataset.
2. Trong **Settings → Accelerator**, chọn GPU T4 hoặc P100.
3. Bật Internet nếu muốn tải trọng số ImageNet.
4. Chạy notebook từ trên xuống. Không chạy Cell train trước các Cell chuẩn bị dữ liệu.

Kết quả sẽ được lưu tại `/kaggle/working/banana_results`.

In [ ]:
print("Kaggle đã chạy")

## Cell 1 — Kiểm tra GPU và khai báo thư viện

In [ ]:
!nvidia-smi

import os
import re
import gc
import json
import time
import shutil
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## Cell 2 — Liệt kê dataset đã thêm vào Kaggle

In [ ]:
INPUT_ROOT = Path("/kaggle/input")

dataset_folders = sorted([p for p in INPUT_ROOT.iterdir() if p.is_dir()])

print("Các dataset trong /kaggle/input:")
for index, folder in enumerate(dataset_folders):
    print(f"[{index}] {folder}")

if not dataset_folders:
    raise RuntimeError(
        "Không tìm thấy dataset. Hãy nhấn Add Input ở bên phải notebook rồi chạy lại cell."
    )


In [ ]:
!find /kaggle/input/datasets -maxdepth 3 -type d | head -50

## Cell 3 — Chọn dataset

Notebook ưu tiên tự tìm dataset có từ `banana` trong tên. Nếu có nhiều dataset chuối, hãy sửa `DATASET_INDEX` theo danh sách được in ra.

In [ ]:
SOURCE_DIR = Path(
    "/kaggle/input/datasets/deeuk1985/"
    "banana-ripeness-rgb-images-and-spectral-data/"
    "banana_ripeness"
)

print("Dataset được chọn:", SOURCE_DIR)
print("Đường dẫn tồn tại:", SOURCE_DIR.exists())

## Cell 4 — Tự tìm thư mục chứa các lớp ảnh RGB

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def count_images(folder):
    return sum(
        1
        for file in Path(folder).rglob("*")
        if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS
    )


def normalize_name(name):
    return re.sub(r"[^a-z0-9]+", " ", name.lower()).strip()


def infer_standard_label(folder_name):
    name = normalize_name(folder_name)

    if any(key in name for key in ["past ripe", "overripe", "over ripe", "too ripe"]):
        return "qua_chin"
    if any(key in name for key in ["under ripe", "underripe", "unripe", "raw"]):
        return "xanh"
    if name == "ripe" or "yellow ripe" in name:
        return "chin"
    return None


candidates = []

folders_to_check = [SOURCE_DIR] + [p for p in SOURCE_DIR.rglob("*") if p.is_dir()]

for folder in folders_to_check:
    try:
        subfolders = [p for p in folder.iterdir() if p.is_dir() and count_images(p) > 0]
    except PermissionError:
        continue

    mapped = [(p, infer_standard_label(p.name)) for p in subfolders]
    valid_labels = {label for _, label in mapped if label is not None}

    if valid_labels == {"xanh", "chin", "qua_chin"}:
        total = sum(count_images(p) for p, label in mapped if label is not None)
        candidates.append((folder, mapped, total))

candidates.sort(key=lambda item: item[2], reverse=True)

if not candidates:
    print("Không tự tìm thấy thư mục có đủ ba lớp.")
    print("Cấu trúc thư mục cấp đầu:")
    for root, dirs, files in os.walk(SOURCE_DIR):
        level = str(root).replace(str(SOURCE_DIR), "").count(os.sep)
        if level <= 4:
            print("    " * level + Path(root).name + f"/ ({len(files)} files)")
    raise RuntimeError(
        "Dataset chưa có cấu trúc ba thư mục lớp. Hãy xem cây thư mục được in ra."
    )

print("Các thư mục phù hợp:")
for index, (folder, mapped, total) in enumerate(candidates):
    print(f"\n[{index}] {folder} — {total} ảnh")
    for class_folder, label in mapped:
        if label is not None:
            print(f"    {class_folder.name} -> {label}: {count_images(class_folder)} ảnh")

# Thông thường chọn 0 vì đây là thư mục có nhiều ảnh hợp lệ nhất.
CANDIDATE_INDEX = 0
CLASS_SOURCE_DIR, CLASS_MAPPING, _ = candidates[CANDIDATE_INDEX]

print("\nThư mục lớp được chọn:", CLASS_SOURCE_DIR)

## Cell 5 — Chuẩn hóa và chia dữ liệu một lần

Cell này tạo đúng một tập train, validation và test với tỷ lệ `70%–15%–15%`. Tất cả năm mô hình sẽ dùng chung các tập này.

In [ ]:
TASK_NAME = "ripeness"
DATASET_DIR = Path("/kaggle/working/banana_dataset_split")
RESULT_DIR = Path("/kaggle/working/banana_results")
MODEL_DIR = RESULT_DIR / "models"
FIGURE_DIR = RESULT_DIR / "figures"

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

records = []

for source_class_folder, standard_label in CLASS_MAPPING:
    if standard_label is None:
        continue

    for image_path in source_class_folder.rglob("*"):
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
            records.append({"path": image_path, "label": standard_label})

records_df = pd.DataFrame(records)

print("Số ảnh theo lớp trước khi chia:")
print(records_df["label"].value_counts())

if records_df["label"].nunique() != 3:
    raise RuntimeError("Dữ liệu sau chuẩn hóa không đủ ba lớp.")

train_df, temp_df = train_test_split(
    records_df,
    test_size=0.30,
    random_state=SEED,
    stratify=records_df["label"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"],
)


def copy_split(dataframe, split_name):
    for row_number, row in dataframe.reset_index(drop=True).iterrows():
        destination_folder = DATASET_DIR / split_name / row["label"]
        destination_folder.mkdir(parents=True, exist_ok=True)

        source_path = Path(row["path"])
        destination_name = f"{row_number:06d}_{source_path.name}"
        shutil.copy2(source_path, destination_folder / destination_name)


copy_split(train_df, "train")
copy_split(val_df, "val")
copy_split(test_df, "test")

print("\nĐã chia dữ liệu:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Đường dẫn:", DATASET_DIR)

## Cell 6 — Đọc dataset và kiểm tra ảnh

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / "train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / "val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / "test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

class_names = train_ds.class_names
print("Thứ tự lớp:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for index in range(min(9, len(images))):
        plt.subplot(3, 3, index + 1)
        plt.imshow(images[index].numpy().astype("uint8"))
        plt.title(class_names[int(labels[index])])
        plt.axis("off")

plt.tight_layout()
plt.show()

## Cell 7 — Khai báo năm mô hình

Nếu Kaggle không thể tải trọng số ImageNet, hãy bật Internet trong Settings. Không nên so sánh mô hình khi một số mô hình dùng ImageNet còn số khác train từ đầu.

In [ ]:
from tensorflow.keras import layers, Model

MODEL_CONFIGS = {
    "MobileNetV3Small": (
        tf.keras.applications.MobileNetV3Small,
        tf.keras.applications.mobilenet_v3.preprocess_input,
    ),
    "EfficientNetB0": (
        tf.keras.applications.EfficientNetB0,
        tf.keras.applications.efficientnet.preprocess_input,
    ),
    "ResNet50": (
        tf.keras.applications.ResNet50,
        tf.keras.applications.resnet50.preprocess_input,
    ),
    "DenseNet121": (
        tf.keras.applications.DenseNet121,
        tf.keras.applications.densenet.preprocess_input,
    ),
    "InceptionV3": (
        tf.keras.applications.InceptionV3,
        tf.keras.applications.inception_v3.preprocess_input,
    ),
}


def build_model(model_name, number_of_classes):
    constructor, preprocess_input = MODEL_CONFIGS[model_name]

    augmentation = tf.keras.Sequential(
        [
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.08),
            layers.RandomZoom(0.10),
            layers.RandomTranslation(0.05, 0.05),
            layers.RandomContrast(0.12),
        ],
        name="augmentation",
    )

    base_model = constructor(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3),
    )
    base_model.trainable = False

    inputs = layers.Input(shape=(224, 224, 3))
    x = augmentation(inputs)
    x = preprocess_input(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(number_of_classes, activation="softmax")(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model, base_model

## Cell 8 — Hàm train, fine-tune và đánh giá

In [ ]:
def train_and_evaluate(model_name):
    print("\n" + "=" * 72)
    print("ĐANG TRAIN:", model_name)
    print("=" * 72)

    tf.keras.backend.clear_session()
    gc.collect()

    model, base_model = build_model(model_name, len(class_names))

    stage1_weights = MODEL_DIR / f"{TASK_NAME}_{model_name}_stage1.weights.h5"
    stage2_weights = MODEL_DIR / f"{TASK_NAME}_{model_name}_stage2.weights.h5"
    final_model_path = MODEL_DIR / f"{TASK_NAME}_{model_name}.keras"

    callbacks_stage1 = [
        tf.keras.callbacks.ModelCheckpoint(
            str(stage1_weights),
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=True,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=4, restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.2, patience=2, min_lr=1e-7, verbose=1
        ),
    ]

    start_time = time.time()

    history1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=12,
        callbacks=callbacks_stage1,
        verbose=1,
    )

    model.load_weights(stage1_weights)
    stage1_val_loss = model.evaluate(val_ds, verbose=0)[0]

    base_model.trainable = True
    for layer in base_model.layers[:-30]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    callbacks_stage2 = [
        tf.keras.callbacks.ModelCheckpoint(
            str(stage2_weights),
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=True,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=4, restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.2, patience=2, min_lr=1e-7, verbose=1
        ),
    ]

    history2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=8,
        callbacks=callbacks_stage2,
        verbose=1,
    )

    model.load_weights(stage2_weights)
    stage2_val_loss = model.evaluate(val_ds, verbose=0)[0]

    if stage1_val_loss <= stage2_val_loss:
        model.load_weights(stage1_weights)
        selected_stage = "frozen_backbone"
    else:
        model.load_weights(stage2_weights)
        selected_stage = "fine_tuned"

    training_time = time.time() - start_time

    test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)

    y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
    probabilities = model.predict(test_ds, verbose=0)
    y_pred = np.argmax(probabilities, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0,
    )
    print(report)

    with open(
        RESULT_DIR / f"{TASK_NAME}_{model_name}_classification_report.txt",
        "w",
        encoding="utf-8",
    ) as file:
        file.write(report)

    matrix = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
    )
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.title(f"Confusion Matrix — {model_name}")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"{TASK_NAME}_{model_name}_confusion.png", dpi=300)
    plt.show()
    plt.close()

    accuracy_values = history1.history["accuracy"] + history2.history["accuracy"]
    val_accuracy_values = history1.history["val_accuracy"] + history2.history["val_accuracy"]
    loss_values = history1.history["loss"] + history2.history["loss"]
    val_loss_values = history1.history["val_loss"] + history2.history["val_loss"]
    epochs = range(1, len(accuracy_values) + 1)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, accuracy_values, label="Train")
    plt.plot(epochs, val_accuracy_values, label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"Accuracy — {model_name}")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss_values, label="Train")
    plt.plot(epochs, val_loss_values, label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Loss — {model_name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"{TASK_NAME}_{model_name}_learning_curves.png", dpi=300)
    plt.show()
    plt.close()

    try:
        model.save(final_model_path)
    except Exception as error:
        print("Không lưu được model .keras, nhưng file trọng số vẫn còn:", error)

    result = {
        "Task": TASK_NAME,
        "Model": model_name,
        "Selected_Stage": selected_stage,
        "Accuracy": float(test_accuracy),
        "Macro_Precision": float(precision),
        "Macro_Recall": float(recall),
        "Macro_F1": float(f1),
        "Test_Loss": float(test_loss),
        "Training_Time_Seconds": float(training_time),
    }

    del model, base_model
    tf.keras.backend.clear_session()
    gc.collect()

    return result

## Cell 9 — Chạy thử một mô hình

Nên chạy Cell này trước. Nếu EfficientNetB0 chạy thành công thì mới chạy cả năm mô hình ở Cell kế tiếp.

In [ ]:
test_result = train_and_evaluate("EfficientNetB0")
pd.DataFrame([test_result])

## Cell 10 — Chạy đủ năm mô hình

Cell này có thể mất nhiều thời gian. Nếu đã chạy EfficientNetB0 ở Cell 9, Cell này sẽ chạy lại EfficientNetB0 để tạo một bảng kết quả thống nhất.

In [ ]:
MODEL_NAMES = [
    "MobileNetV3Small",
    "EfficientNetB0",
    "ResNet50",
    "DenseNet121",
    "InceptionV3",
]

all_results = []

for model_name in MODEL_NAMES:
    try:
        result = train_and_evaluate(model_name)
        all_results.append(result)

        pd.DataFrame(all_results).to_csv(
            RESULT_DIR / "five_models_results.csv", index=False
        )
    except Exception as error:
        print(f"LỖI Ở {model_name}: {type(error).__name__}: {error}")

results_df = pd.DataFrame(all_results)

if results_df.empty:
    raise RuntimeError("Không có mô hình nào train thành công.")

results_df = results_df.sort_values("Macro_F1", ascending=False).reset_index(drop=True)
results_df.to_csv(RESULT_DIR / "five_models_results.csv", index=False)
results_df

## Cell 11 — Biểu đồ so sánh năm mô hình

In [ ]:
plot_data = results_df.melt(
    id_vars="Model",
    value_vars=["Accuracy", "Macro_Precision", "Macro_Recall", "Macro_F1"],
    var_name="Metric",
    value_name="Score",
)

plt.figure(figsize=(13, 7))
sns.barplot(data=plot_data, x="Model", y="Score", hue="Metric")
plt.ylim(0, 1.05)
plt.xlabel("Model")
plt.ylabel("Score")
plt.title("Comparison of Five Models — Banana Ripeness")
plt.xticks(rotation=15)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "five_models_comparison.png", dpi=300)
plt.show()

best_row = results_df.iloc[0]
print("Mô hình tốt nhất:", best_row["Model"])
print("Accuracy:", round(best_row["Accuracy"], 4))
print("Macro F1:", round(best_row["Macro_F1"], 4))

## Cell 12 — Nén toàn bộ kết quả để tải về

In [ ]:
archive_path = shutil.make_archive(
    "/kaggle/working/banana_results",
    "zip",
    root_dir=RESULT_DIR,
)

print("Đã tạo:", archive_path)
print("Mở mục Output của Kaggle để tải banana_results.zip về máy.")

## Ghi chú quan trọng

- Dataset này chỉ dùng cho bài toán **Xanh – Chín – Quá chín**.
- Để train **Tốt – Đốm – Thối**, cần chuẩn bị dataset khác có ba thư mục `tot`, `dom`, `thoi`, rồi thay phần ánh xạ nhãn.
- Không chia lại train/validation/test cho từng mô hình; notebook đã cố định `SEED = 42` và chia đúng một lần.
- Khi so sánh mô hình, ưu tiên **Macro F1**, đồng thời xem Accuracy, confusion matrix và learning curves.
- Nếu hết GPU hoặc RAM, giảm `BATCH_SIZE` từ `16` xuống `8`.